# Notebook 04 — Resume Builder
**Mission Veteran AI · Veteran Career Navigator**

Tests `build_resume_draft` — loads PT-002, assembles an evidence-bounded
prompt, calls Granite, tags every sentence EVIDENCED / INFERRED / MISSING,
and evaluates the export gate.

**Prerequisites**
- `ibm-watsonx-ai` installed  
- `.env` file with `WATSONX_API_KEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`  
- Notebooks 01–03 completed (or use the sample inputs below)

In [ ]:
%pip install -q ibm-watsonx-ai python-dotenv

In [ ]:
import json, os, re, sys
from dotenv import load_dotenv

load_dotenv()
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

API_KEY    = os.environ['WATSONX_API_KEY']
WX_URL     = os.environ.get('WATSONX_URL', 'https://us-south.ml.cloud.ibm.com')
PROJECT_ID = os.environ['WATSONX_PROJECT_ID']
print('Credentials loaded ✓')

## 1 · Load PT-002 from tblPrompts

In [ ]:
KB_PATH = os.path.join(ROOT, 'knowledge-bases', 'prompt_templates.json')
with open(KB_PATH, 'r', encoding='utf-8') as f:
    templates = json.load(f)

template = next(t for t in templates if t['name'] == 'resume_draft_generation')
print(f"Loaded: {template['template_id']} — {template['description']}")

## 2 · Sample inputs (replace with real bullet translation/2/3 outputs)

In [ ]:
SERVICE_BRANCH   = 'Army'
SPECIALTY_CODE   = '92A'
SERVICE_SUMMARY  = (
    'Managed automated supply logistics for a 500-person organization over 4 years, '
    'overseeing a $2.1M annual inventory. Led a 12-member team, improved on-time '
    'delivery from 82% to 96%, and implemented a cycle-count program that reduced '
    'shrinkage by 18%.'
)
TARGET_ROLE = 'Supply Chain Operations Manager'

TRANSLATED_BULLETS = json.dumps([
    'Directed a 12-member logistics team, improving on-time delivery rates from 82% to 96% over 18 months.',
    'Managed a $2.1M annual inventory across multiple warehouse locations, achieving a 98.7% accountability rate.',
    'Implemented a cycle-count reconciliation program that reduced inventory shrinkage by 18%.',
])

SKILL_INVENTORY = json.dumps([
    {'skill_name': 'logistics coordination',        'category': 'Operations',    'confidence': 'High'},
    {'skill_name': 'team leadership',               'category': 'Leadership',    'confidence': 'High'},
    {'skill_name': 'inventory management',          'category': 'Supply Chain',  'confidence': 'High'},
    {'skill_name': 'budget oversight',              'category': 'Finance',       'confidence': 'Medium'},
    {'skill_name': 'cross-functional coordination', 'category': 'Collaboration', 'confidence': 'High'},
])

JOB_ANALYSIS = json.dumps({
    'role_title': 'Supply Chain Operations Manager',
    'seniority_level': 'Mid-level (3-5 years)',
    'industry': 'Logistics / Distribution',
    'required_skills': [
        'supply chain coordination', 'inventory management', 'vendor management',
        'budget oversight', 'ERP/WMS proficiency', 'cross-functional team leadership',
    ],
    'preferred_skills': ['APICS CSCP', 'Six Sigma', 'SAP'],
    'ats_keywords': ['supply chain', 'logistics', 'inventory', 'KPI', 'cross-functional'],
})

print('Sample inputs defined ✓')

## 3 · Assemble evidence-bounded prompt from PT-002

In [ ]:
bullets_list    = json.loads(TRANSLATED_BULLETS)
skill_inventory = json.loads(SKILL_INVENTORY)
job_analysis    = json.loads(JOB_ANALYSIS)

bullets_text = '\n'.join(f'- {b}' for b in bullets_list)
skills_text  = '\n'.join(
    f"- {s['skill_name']} ({s['category']}, {s['confidence']})"
    for s in skill_inventory
)
job_text = json.dumps(job_analysis, indent=2)

user_prompt = template['user_prompt_template'].format(
    service_branch=SERVICE_BRANCH,
    specialty_code=SPECIALTY_CODE,
    service_summary=SERVICE_SUMMARY[:1000],
    translated_bullets=bullets_text[:800],
    skill_inventory=skills_text[:600],
    job_analysis=job_text[:600],
    target_role=TARGET_ROLE,
)

print('Prompt assembled. Length:', len(user_prompt), 'chars')

## 4 · Call IBM Granite

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = 'ibm/granite-4-h-small'
p        = template['model_params']

model = ModelInference(
    model_id=MODEL_ID,
    api_client=APIClient(credentials=Credentials(url=WX_URL, api_key=API_KEY)),
    project_id=PROJECT_ID,
    params={
        Params.MAX_NEW_TOKENS:    p['max_new_tokens'],
        Params.MIN_NEW_TOKENS:    p['min_new_tokens'],
        Params.TEMPERATURE:       p['temperature'],
        Params.REPETITION_PENALTY:p['repetition_penalty'],
    },
)

response = model.generate(
    prompt=f"System:\n{template['system_prompt']}\nUser:\n{user_prompt}\nAssistant:\n",
)

raw = response['results'][0]['generated_text'].strip()
raw = re.sub(r'^```[a-z]*\n?', '', raw)
raw = re.sub(r'\n?```$', '', raw)
print('Raw output length:', len(raw), 'chars')
print(raw[:500], '...')

## 5 · Parse output and tag every sentence

In [ ]:
draft = json.loads(raw)

print('═' * 60)
print('SUMMARY')
summary = draft.get('summary', {})
if isinstance(summary, str):
    summary = {'text': summary, 'tag': 'INFERRED'}
tag_icon = {'EVIDENCED': '✓', 'INFERRED': '⚠', 'MISSING': '✗'}
print(f"  [{summary.get('tag','?')}] {tag_icon.get(summary.get('tag','?'),'?')}  {summary.get('text','')}")

print()
print('BULLETS')
for b in draft.get('bullets', []):
    icon = tag_icon.get(b.get('tag', '?'), '?')
    print(f"  [{b.get('tag','?')}] {icon}  {b.get('text','')}")

print()
print('SKILLS SECTION')
for s in draft.get('skills_section', []):
    print(f'  • {s}')

## 6 · Gate evaluation

In [ ]:
all_tagged = draft.get('bullets', []) + [summary]
counts = {'EVIDENCED': 0, 'INFERRED': 0, 'MISSING': 0}
for item in all_tagged:
    t = item.get('tag', '').upper()
    if t in counts:
        counts[t] += 1

gate_status = 'OPEN' if counts['INFERRED'] == 0 and counts['MISSING'] == 0 else 'BLOCKED'

print(f'Tag counts: {counts}')
print(f'Gate status: {gate_status}')

if gate_status == 'BLOCKED':
    print()
    print('Blocked sentences:')
    for item in all_tagged:
        if item.get('tag', '').upper() in ('INFERRED', 'MISSING'):
            print(f"  [{item['tag']}] {item.get('text','')}")